# CDD-11-30: A3 degradation-supervision calibration

A3 keeps the selected A2 bottleneck + skip-gate architecture and changes only `degradation_weight` from 0 to 0.05. It therefore tests whether multi-label low/haze/rain/snow supervision improves restoration and degradation reasoning. The run uses SIDD32, seed 42, five epochs, two T4 GPUs through DDP, and full-frame validation/evaluation. Expected runtime is approximately 7–12 minutes.

Only the existing `cdd-11-30` and `nafnetmodel` Kaggle inputs are required.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_a3")
CONFIG = Path("configs/calibration_a3.json")
RUN_NAME = "a3_degradation_supervision_sidd32_seed42_5ep"
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
], text=True).strip().splitlines()
assert len(gpu_names) == 2, f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
print("GPUs:", gpu_names)
print("CDD-11:", CDD11_ROOT)
print("Pretrained files:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Recheck data pairs, leakage, and exact pretrained compatibility.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_a3.json",
], check=True)

In [ ]:
# Set True only after the audit above succeeds.
RUN_A3 = False

In [ ]:
runner_command = [
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--runs", RUN_NAME,
]
if RUN_A3:
    subprocess.run(runner_command, check=True)
else:
    subprocess.run([*runner_command, "--dry-run"], check=True)
    print("Dry run complete. Set RUN_A3 = True and rerun from the safety-switch cell.")

In [ ]:
# Display metrics and create separate lightweight and qualitative downloads.
import zipfile
from IPython.display import FileLink, FileLinks, display

summary_csv = EXPERIMENTS_ROOT / "ablation_summary.csv"
if summary_csv.is_file():
    import pandas as pd
    display(pd.read_csv(summary_csv))

    lightweight = Path("/kaggle/working/a3_lightweight_results.zip")
    with zipfile.ZipFile(lightweight, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(EXPERIMENTS_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".pt", ".png", ".jpg", ".jpeg"}:
                output_zip.write(path, path.relative_to(EXPERIMENTS_ROOT))

    comparisons = Path("/kaggle/working/a3_comparisons.zip")
    comparison_root = EXPERIMENTS_ROOT / RUN_NAME / "evaluation" / "comparisons"
    with zipfile.ZipFile(comparisons, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(comparison_root.glob("*.png")):
            output_zip.write(path, path.name)

    print("Download both files and send them back for A2/A3 analysis:")
    display(FileLink(str(lightweight)))
    display(FileLink(str(comparisons)))
else:
    print("No completed A3 summary yet.")

if EXPERIMENTS_ROOT.is_dir():
    display(FileLinks(str(EXPERIMENTS_ROOT)))